# 阶段 1：金融数据获取与清洗

## 研究对象说明

- **标的**: SPY (SPDR S&P 500 ETF Trust)，追踪标普 500 指数的交易所交易基金
- **时间范围**: 2015-01-01 至 2026-01-01（实际数据覆盖至 2025-12-31）
- **数据频率**: 日线
- **字段说明**:
  - Open: 开盘价
  - High: 最高价
  - Low: 最低价
  - Close: 收盘价
  - Adj Close: 复权收盘价
  - Volume: 成交量

数据来源: Yahoo Finance (yfinance)

In [ ]:
import pandas as pd
from pathlib import Path

# 探测项目根目录
cwd = Path.cwd()
if (cwd / "data").exists() and (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists() and (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "未找到项目根目录，请从项目根目录或 notebooks 目录启动 Notebook。"
    )

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "SPY_raw_2015_2025.csv"
CLEAN_PATH = PROJECT_ROOT / "data" / "processed" / "SPY_clean_2015_2025.csv"
REPORT_PATH = PROJECT_ROOT / "outputs" / "tables" / "SPY_data_quality_report.csv"
FIG_DIR = PROJECT_ROOT / "outputs" / "figures"

# 读取原始数据
if RAW_PATH.exists():
    df_raw = pd.read_csv(RAW_PATH, index_col=0, parse_dates=True)
    print(f'原始数据加载成功: {df_raw.shape[0]} 行, {df_raw.shape[1]} 列')
    print(f'日期范围: {df_raw.index.min().strftime("%Y-%m-%d")} ~ {df_raw.index.max().strftime("%Y-%m-%d")}')
    display(df_raw.head())
else:
    print(f'文件不存在: {RAW_PATH.resolve()}')
    print('请先运行: python run_stage1.py')


In [ ]:
# 读取清洗后数据
if CLEAN_PATH.exists():
    df_clean = pd.read_csv(CLEAN_PATH, index_col=0, parse_dates=True)
    print('=== 基本信息 ===')
    print(f'shape: {df_clean.shape}')
    print(f'columns: {list(df_clean.columns)}')
    print('dtypes:')
    print(df_clean.dtypes)
    print(f'日期范围: {df_clean.index.min().strftime("%Y-%m-%d")} ~ {df_clean.index.max().strftime("%Y-%m-%d")}')
    print()
    print('=== 前5行 ===')
    display(df_clean.head())
else:
    print(f'文件不存在: {CLEAN_PATH.resolve()}')
    print('请先运行: python run_stage1.py')


In [ ]:
if CLEAN_PATH.exists():
    print('=== 缺失值统计 ===')
    missing = df_clean.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0])
    else:
        print('无缺失值')
    print()
    print('=== 重复日期检查 ===')
    n_dup = df_clean.index.duplicated().sum()
    print(f'重复日期数: {n_dup}')
    print()
    print('=== 描述性统计 ===')
    display(df_clean.describe())


In [ ]:
# 读取质量报告
if REPORT_PATH.exists():
    df_report = pd.read_csv(REPORT_PATH)
    print(f'质量报告共 {len(df_report)} 项指标')
    print()
    key_items = ['ticker', '数据总行数', '起始日期', '结束日期', '日期是否单调递增', '是否存在重复日期']
    for item in key_items:
        mask = df_report['指标'] == item
        if mask.any():
            val = df_report.loc[mask, '值'].values[0]
            print(f'{item}: {val}')
    print()
    key_items2 = ['return_1d_均值', 'return_1d_标准差', 'Close_最小值', 'Close_最大值', 'Volume_最小值', 'Volume_最大值']
    for item in key_items2:
        mask = df_report['指标'] == item
        if mask.any():
            val = df_report.loc[mask, '值'].values[0]
            print(f'{item}: {val}')
else:
    print(f'文件不存在: {REPORT_PATH.resolve()}')
    print('请先运行: python run_stage1.py')


In [ ]:
# 展示数据概览图
from IPython.display import Image, display

figures = {
    'SPY 收盘价 + 移动均线': 'SPY_close_price.png',
    'SPY 成交量': 'SPY_volume.png',
    'SPY 日收益率分布': 'SPY_return_distribution.png',
}

for title, fname in figures.items():
    fpath = FIG_DIR / fname
    if fpath.exists():
        print(f'### {title}')
        display(Image(filename=str(fpath)))
    else:
        print(f'### {title}')
        print(f'图片不存在: {fpath.resolve()}')
        print()


## 阶段 1 小结

- SPY 历史行情数据已成功获取，覆盖 2015-01-02 至 2025-12-31，共 2766 个交易日
- 数据清洗完成：无缺失值、无重复日期、日期单调递增、数据类型已标准化
- 数据质量报告已生成，各项指标正常
- 三张基础可视化图表已完成：收盘价走势、成交量变化、日收益率分布
- 阶段 1 数据获取、清洗、质量检查与基础可视化已经完成，后续实验可基于本阶段生成的清洗数据继续开展。